In [1]:
import os
import numpy as np
import pandas as pd
import astropy.units as u
import matplotlib.pyplot as plt

from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.simbad import Simbad
from matplotlib.patches import Rectangle
from astroquery.hips2fits import hips2fits
from astropy.coordinates import SkyCoord, Angle

font = {'family' : 'gill sans',
         'size'   : 12,
         'serif':  'cmr10'
         }

plt.rc('font', **font)
plt.rc('axes', unicode_minus = False)

In [2]:
main_directory = Path().resolve()

In [3]:
def parse_directory(dirname, galaxy_name, region_name, target_name, angular_separation): # different with some other functions I wrote

    parts = dirname.split("_")

    Grating = parts[0]
    Cenwave = int(parts[1])
    T_exp = str(parts[3])
    PID = int(parts[4])

    if Grating.startswith("G"):
        
        Instrument = "COS"
        LP = parts[2]

        return {
            # "Galaxy": galaxy_name, 
            "Region": region_name, 
            "Target": target_name, 
            "Angular Separation": round(angular_separation, 2), 
            "Instrument/Grating/Cenwave": Instrument + "/" + Grating + "/" + str(Cenwave), 
            "LP/Aperture": LP,  
            "T_exp": str(T_exp), 
            "PID": str(PID), 
        }
        
    else:
        
        Instrument = "STIS"
        Aperture = parts[2]
        return {
            # "Galaxy": galaxy_name, 
            "Region": region_name, 
            "Target": target_name, 
            "Angular Separation": round(angular_separation, 2), 
            "Instrument/Grating/Cenwave": Instrument + "/" + Grating + "/" + str(Cenwave), 
            "LP/Aperture": Aperture, 
            "T_exp": str(T_exp), 
            "PID": str(PID), 
        }

In [4]:
records = []

for galaxy_name in os.listdir(main_directory):
    galaxy_path = main_directory / galaxy_name
    if galaxy_path.name.startswith(".") or not galaxy_path.is_dir():
        continue
        
    for region_name in os.listdir(galaxy_path):
        region_path = galaxy_path / region_name
        if region_path.name.startswith(".") or not region_path.is_dir():
            continue

        region = Simbad.query_object(region_name)
        ra_region = region["ra"][0]
        dec_region = region["dec"][0]
        coord_region = SkyCoord(ra_region, dec_region, unit = (u.deg, u.deg), frame = "icrs")
    
        for target_name in os.listdir(region_path):
            target_path = region_path / target_name
            if target_path.name.startswith("unused") or target_path.name.startswith(".") or not target_path.is_dir():
                continue

            target = Simbad.query_object(target_name)
            ra_target = target["ra"][0]
            dec_target = target["dec"][0]
            coord_target = SkyCoord(ra_target, dec_target, unit = (u.deg, u.deg), frame = "icrs")

            angular_separation = coord_region.separation(coord_target).arcmin

            spectra_directories = sorted([d for d in os.listdir(target_path) if d.startswith("G") or d.startswith("E")])

            for d in spectra_directories:
                info = parse_directory(d, galaxy_name, region_name, target_name, angular_separation)
                records.append(info)

df = pd.DataFrame.from_records(records)

In [5]:
df

,Region,Target,Angular Separation,Instrument/Grating/Cenwave,LP/Aperture,T_exp,PID
0,SMC N88A,2dFS 3694,1.93,COS/G130M/1291,LP4,1534.400,16101
1,SMC N88A,2dFS 3694,1.93,COS/G160M/1611,LP4,2336.032,16101
2,SMC N90,SK 183,0.41,STIS/E140M/1425,0.2x0.2,2707.000,15837
3,SMC N90,SK 183,0.41,COS/G130M/1096,LP2,1736.064,16808
4,SMC N90,SK 183,0.41,COS/G130M/1096,LP2,1736.096,17295
...,...,...,...,...,...,...,...
77,30 Doradus,BAT99 114,0.20,COS/G160M/1611,LP6,2024.096,16812
78,30 Doradus,BAT99 113,0.28,COS/G130M/1291,LP5,2062.400,16812
79,30 Doradus,BAT99 113,0.28,COS/G160M/1611,LP6,2004.032,16812
80,30 Doradus,Cl* NGC 2070 MEL 55,1.86,COS/G130M/1291,LP5,4774.368,16815


In [6]:
df.to_csv("table.csv", index = False)

In [7]:
latex = df.to_latex(index = False, header = False, escape = True, na_rep = "/", float_format = "%.2f")
lines = latex.splitlines()
lines = "\n".join(lines)
print(lines)

\begin{tabular}{llrllll}
\toprule
\midrule
SMC N88A & 2dFS 3694 & 1.93 & COS/G130M/1291 & LP4 & 1534.400 & 16101 \\
SMC N88A & 2dFS 3694 & 1.93 & COS/G160M/1611 & LP4 & 2336.032 & 16101 \\
SMC N90 & SK 183 & 0.41 & STIS/E140M/1425 & 0.2x0.2 & 2707.000 & 15837 \\
SMC N90 & SK 183 & 0.41 & COS/G130M/1096 & LP2 & 1736.064 & 16808 \\
SMC N90 & SK 183 & 0.41 & COS/G130M/1096 & LP2 & 1736.096 & 17295 \\
SMC N66A & Cl* NGC 346 ELS 51 & 0.53 & COS/G130M/1096 & LP2 & 9959.808 & 16103 \\
SMC N66A & Cl* NGC 346 ELS 51 & 0.53 & COS/G130M/1291 & LP1 & 360.032 & 11625 \\
SMC N66A & Cl* NGC 346 ELS 51 & 0.53 & COS/G130M/1327 & LP1 & 360.032 & 11625 \\
SMC N66A & Cl* NGC 346 ELS 51 & 0.53 & COS/G160M/1577 & LP1 & 648.224 & 11625 \\
SMC N66A & Cl* NGC 346 ELS 51 & 0.53 & COS/G160M/1623 & LP1 & 648.224 & 11625 \\
SMC N66A & Cl* NGC 346 ELS 50 & 1.69 & COS/G130M/1096 & LP2 & 9965.762 & 16103 \\
SMC N66A & Cl* NGC 346 ELS 50 & 1.69 & COS/G130M/1291 & LP1 & 335.008 & 11625 \\
SMC N66A & Cl* NGC 346 ELS 50 

In [8]:
region_order = [
    "SMC N66A",
    "SMC N81",
    "SMC N88A",
    "SMC N90",
    "30 Doradus",
    "LMC N11B",
]

df["Region"] = pd.Categorical(df["Region"], categories = region_order, ordered = True)

df = df.sort_values(
    by = ["Region", "Target", "Angular Separation"]
).reset_index(drop = True)

In [17]:
cols_to_merge = ["Region", "Target", "Angular Separation"]

mask = df.duplicated(subset = cols_to_merge)

df_merged = df.copy()

for c in cols_to_merge:
    df_merged[c] = df_merged[c].astype("string")

df_merged.loc[mask, cols_to_merge] = ""

In [18]:
def df_to_latex_with_target_hlines(df):
    lines = []
    prev_target = None

    for _, row in df.iterrows():
        curr_target = row["Target"]
        if prev_target is not None and curr_target != "" and curr_target != prev_target:
            lines.append(r"\hline")
        line = " & ".join(
            "" if pd.isna(v) or v == "" else f"{v:.2f}" if isinstance(v, float) else str(v)
            for v in row
        ) + r" \\"
        lines.append(line)
        prev_target = curr_target if curr_target != "" else prev_target

    return "\n".join(lines)

In [19]:
body = df_to_latex_with_target_hlines(df_merged)

latex = rf"""
\begin{{tabular}}{{l l r l l r r}}
\hline
{body}
\hline
\end{{tabular}}
"""

print(latex)


\begin{tabular}{l l r l l r r}
\hline
SMC N66A & Cl* NGC 346 ELS 22 & 1.2 & COS/G130M/1291 & LP1 & 360.000 & 11625 \\
 &  &  & COS/G130M/1327 & LP1 & 360.032 & 11625 \\
 &  &  & COS/G160M/1577 & LP1 & 560.032 & 11625 \\
 &  &  & COS/G160M/1623 & LP1 & 560.032 & 11625 \\
\hline
SMC N66A & Cl* NGC 346 ELS 50 & 1.69 & COS/G130M/1096 & LP2 & 9965.762 & 16103 \\
 &  &  & COS/G130M/1291 & LP1 & 335.008 & 11625 \\
 &  &  & COS/G130M/1327 & LP1 & 334.016 & 11625 \\
 &  &  & COS/G160M/1623 & LP1 & 600.192 & 11625 \\
\hline
SMC N66A & Cl* NGC 346 ELS 51 & 0.53 & COS/G130M/1096 & LP2 & 9959.808 & 16103 \\
 &  &  & COS/G130M/1291 & LP1 & 360.032 & 11625 \\
 &  &  & COS/G130M/1327 & LP1 & 360.032 & 11625 \\
 &  &  & COS/G160M/1577 & LP1 & 648.224 & 11625 \\
 &  &  & COS/G160M/1623 & LP1 & 648.224 & 11625 \\
\hline
SMC N66A & Cl* NGC 346 ELS 7 & 0.54 & STIS/E140M/1425 & 0.2x0.2 & 5400.000 & 7437 \\
\hline
SMC N66A & Cl* NGC 346 MPG 782 & 2.49 & COS/G130M/1096 & LP2 & 10611.616 & 16371 \\
 &  &  & C

In [ ]:
# END